# Feature Engineering: California Housing

This notebook demonstrates key feature engineering techniques and measures their impact on model performance.

**Techniques covered:**
1. Polynomial features
2. Binning / discretization
3. Log transforms
4. Interaction features
5. Before/after model comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

sns.set_style('whitegrid')
%matplotlib inline

## Load Data

In [ ]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

print(f"Shape: {df.shape}")
df.describe().round(3)

## Baseline Model

We start with a Ridge regression on raw features to establish a baseline R² score.

In [ ]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = Ridge(alpha=1.0)
baseline_scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring='r2')
print(f"Baseline Ridge R²: {baseline_scores.mean():.4f} (+/- {baseline_scores.std():.4f})")

## 1. Log Transform

Skewed features can benefit from log transformation to reduce the impact of outliers and make distributions more symmetric.

In [ ]:
skewed_cols = ['MedInc', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']

fig, axes = plt.subplots(2, len(skewed_cols), figsize=(18, 7))
for i, col in enumerate(skewed_cols):
    axes[0, i].hist(X_train[col], bins=50, alpha=0.7)
    axes[0, i].set_title(f'{col} (raw)')
    axes[1, i].hist(np.log1p(X_train[col]), bins=50, alpha=0.7, color='coral')
    axes[1, i].set_title(f'{col} (log1p)')

plt.tight_layout()
plt.show()

X_log = X_train.copy()
for col in skewed_cols:
    X_log[col] = np.log1p(X_log[col])

log_scores = cross_val_score(Ridge(alpha=1.0), X_log, y_train, cv=5, scoring='r2')
print(f"Log-transformed R²: {log_scores.mean():.4f} (+/- {log_scores.std():.4f})")

## 2. Polynomial and Interaction Features

Polynomial features capture non-linear relationships and interactions between variables.

In [ ]:
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
X_poly = poly.fit_transform(X_train)
print(f"Original features: {X_train.shape[1]}")
print(f"Polynomial features: {X_poly.shape[1]}")

poly_scores = cross_val_score(Ridge(alpha=10.0), X_poly, y_train, cv=5, scoring='r2')
print(f"\nPolynomial R²: {poly_scores.mean():.4f} (+/- {poly_scores.std():.4f})")

## 3. Binning / Discretization

Binning converts continuous features into discrete categories. This can help models capture step-function patterns (e.g., house age brackets).

In [ ]:
X_eng = X_train.copy()

# Bin HouseAge into categories
X_eng['AgeGroup'] = pd.cut(X_eng['HouseAge'], bins=[0, 10, 20, 35, 52], labels=[0, 1, 2, 3]).astype(float)

# Bin MedInc into quartiles
X_eng['IncomeGroup'] = pd.qcut(X_eng['MedInc'], q=5, labels=[0, 1, 2, 3, 4]).astype(float)

# Manual interaction: rooms per person
X_eng['RoomsPerPerson'] = X_eng['AveRooms'] / (X_eng['AveOccup'] + 1)

# Log transforms on skewed
for col in skewed_cols:
    X_eng[f'{col}_log'] = np.log1p(X_eng[col])

print(f"Engineered features: {X_eng.shape[1]} (was {X_train.shape[1]})")

eng_scores = cross_val_score(Ridge(alpha=1.0), X_eng.fillna(0), y_train, cv=5, scoring='r2')
print(f"Engineered features R²: {eng_scores.mean():.4f} (+/- {eng_scores.std():.4f})")

## 4. Results Comparison

Let's compare all approaches side by side.

In [ ]:
results = pd.DataFrame({
    'Approach': ['Baseline (raw)', 'Log Transform', 'Polynomial (deg=2)', 'Custom Engineered'],
    'Mean R²': [baseline_scores.mean(), log_scores.mean(), poly_scores.mean(), eng_scores.mean()],
    'Std R²': [baseline_scores.std(), log_scores.std(), poly_scores.std(), eng_scores.std()]
})
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(results['Approach'], results['Mean R²'], yerr=results['Std R²'],
              capsize=5, color=['steelblue', 'coral', 'green', 'purple'], alpha=0.8)
ax.set_ylabel('R² Score (5-fold CV)')
ax.set_title('Feature Engineering: Before vs After Comparison')
ax.set_ylim(0, 1)
for bar, val in zip(bars, results['Mean R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.4f}',
            ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Key Takeaways

- **Log transforms** help with skewed features but may not always improve linear models significantly
- **Polynomial features** can substantially boost performance by capturing non-linear patterns, but increase dimensionality
- **Custom engineered features** (binning, interactions) can provide targeted improvements based on domain knowledge
- Always validate improvements with cross-validation to avoid overfitting